# 컨텍스트 분리와 Isolate

같은 상태에 섞인 여러 업무 메시지에서 현재 업무에 필요한 내용만 모델 요청으로 분리하려면 어떻게 할까요?


## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/08-컨텍스트-분리와-Isolate.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.


## 패키지 버전 확인


In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 메시지 수집 모델 준비


In [2]:
# 이 셀은 메시지 수집 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration,ChatResult
class CapturingModel(BaseChatModel):
    seen:list[str]=[]
    @property
    def _llm_type(self) -> str: return "capture"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        self.seen=[m.text for m in messages]
        return ChatResult(generations=[ChatGeneration(message=AIMessage("확인 완료"))])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "CapturingModel": return self

## 업무가 섞인 메시지 준비

컨텍스트 분리는 한 대화 기록에서 현재 작업에 필요한 메시지만 모델 입력으로 선택하는 처리입니다. `mixed`에는 대출과 급여 두 업무의 메시지가 함께 있고, `CapturingModel.seen`은 모델이 실제로 받은 텍스트를 보관합니다. Before는 다섯 메시지를 모두 전달하므로 현재 질문과 무관한 급여 정보도 모델 입력에 포함됩니다.

`IsolateMiddleware("loan")`은 모델 호출 직전 요청의 메시지를 업무 표식으로 걸러 새 요청을 전달합니다. 원래 그래프 상태를 영구 삭제하는 것과는 다르며, 이번 모델 호출의 가시 범위를 줄입니다. 필터가 필요한 근거까지 제거하면 답변 품질이 낮아질 수 있으므로 업무 ID를 붙이는 시점과 기본 포함 규칙을 명확히 해야 합니다.


In [3]:
# 이 셀은 업무가 섞인 메시지 준비 작업을 수행하고 확인할 값을 출력합니다.
from langchain_core.messages import HumanMessage,AIMessage
mixed=[
 HumanMessage("대출은 5권",additional_kwargs={"topic":"loan"}),AIMessage("기간은 14일",additional_kwargs={"topic":"loan"}),
 HumanMessage("급여 자료는 내부 문서",additional_kwargs={"topic":"hr"}),AIMessage("인사팀 확인",additional_kwargs={"topic":"hr"}),
 HumanMessage("대출 연장은?",additional_kwargs={"topic":"loan"})]
print("state_messages:",len(mixed))

state_messages: 5


## Before: 전체 메시지 전달


In [4]:
# 이 셀은 Before: 전체 메시지 전달 작업을 수행하고 확인할 값을 출력합니다.
before_model=CapturingModel()
create_agent(before_model,tools=[]).invoke({"messages":mixed})
print("model_messages:",before_model.seen)
assert len(before_model.seen)==5 and any("급여" in text for text in before_model.seen)

model_messages: ['대출은 5권', '기간은 14일', '급여 자료는 내부 문서', '인사팀 확인', '대출 연장은?']


## After: 업무별 요청 분리


In [5]:
# 이 셀은 After: 업무별 요청 분리 작업을 수행하고 확인할 값을 출력합니다.
class IsolateMiddleware(AgentMiddleware):
    def __init__(self, topic: str) -> None: self.topic=topic
    def wrap_model_call(self, request: Any, handler: Any) -> Any:
        selected=[m for m in request.messages if m.additional_kwargs.get("topic")==self.topic]
        return handler(request.override(messages=selected))
after_model=CapturingModel()
create_agent(after_model,tools=[],middleware=[IsolateMiddleware("loan")]).invoke({"messages":mixed})
print("model_messages:",after_model.seen)
assert len(after_model.seen)==3 and all("급여" not in text for text in after_model.seen)

model_messages: ['대출은 5권', '기간은 14일', '대출 연장은?']


## 상태와 요청 범위 비교

하나의 고객이 주문과 환불을 동시에 문의하면 업무별 분리가 오히려 연결 정보를 끊을 수 있습니다. 이런 경우 공통 고객 정보는 별도 컨텍스트로 제공하고, 업무별 대화만 선택하는 두 층 구조가 필요합니다. 단순 키워드 필터보다 생성 시점에 명시적 `task_id`를 붙이는 편이 경계를 확인하기 쉽습니다.

격리는 접근 통제와 같지 않습니다. 모델 입력에서 급여 문장을 뺐더라도 원본 상태나 로그를 읽을 권한이 그대로라면 보안 경계가 완성되지 않습니다. 이번 출력은 모델이 받은 다섯 문장이 세 문장으로 줄고 급여 문자열이 사라진 사실만 보여 줍니다.


In [6]:
# 이 셀은 상태와 요청 범위 비교 작업을 수행하고 확인할 값을 출력합니다.
print({"stored_messages":len(mixed),"selected_messages":len(after_model.seen),"security_boundary":False,"estimated_cost_usd":0.0})

{'stored_messages': 5, 'selected_messages': 3, 'security_boundary': False, 'estimated_cost_usd': 0.0}
